# P15 — Optimización directa de preferencias: tu modelo de lenguaje ya es un modelo de recompensa

## 1. Título y paper

**Paper:** *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*  
**Autoría:** Rafael Rafailov, Archit Sharma, Eric Mitchell, Stefano Ermon, Christopher D. Manning, Chelsea Finn  
**Año y venue:** 2023 · arXiv:2305.18290 · NeurIPS 2023  
**Nivel:** L4 · **Motor:** `dpo`  
**Ficha completa:** [`P15_dpo`](../../papers/foundational/P15_dpo/README.md)

**Hito:** Alinear un modelo con preferencias humanas sin modelo de recompensa explícito ni bucle de aprendizaje por refuerzo.

- [arXiv:2305.18290](https://arxiv.org/abs/2305.18290)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: El pipeline RLHF es frágil y caro: entrena un modelo extra, requiere muestreo on-policy y ajustar PPO es delicado.
2. Ejecutar una implementación mínima de la propuesta: Derivar la solución óptima del objetivo RLHF con restricción KL y reescribirlo como una pérdida de clasificación binaria sobre pares de preferencias.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P12


## 4. Intuición

RLHF entrena un juez (modelo de recompensa) y luego entrena al alumno a gustar al juez. DPO demuestra que el alumno **ya contiene** al juez: se puede ajustar directamente con las comparaciones, sin construir el juez aparte.


## 5. Concepto mínimo

El óptimo del objetivo RLHF con restricción KL es `π*(y|x) ∝ π_ref(y|x)·exp(r(x,y)/β)`. Despejando `r` y sustituyendo en Bradley-Terry:

```text
L_DPO = −log σ( β·[ log π(y_w|x)/π_ref(y_w|x) − log π(y_l|x)/π_ref(y_l|x) ] )
```

La recompensa implícita es `r̂ = β·log(π/π_ref)`. No hay RL, no hay muestreo on-policy.


## 6. Código explicado

El motor optimiza la pérdida DPO sobre una política de 3 opciones y muestra la recompensa implícita resultante.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('dpo', seed=7)['result']
show(r['politica_referencia'])
show(r['politica_dpo'])
show(r['recompensa_implicita_beta_log_ratio'])

## 7. Predicción antes de ejecutar

1. ¿Hacia qué opción se desplazará la política tras optimizar?
2. ¿Qué signo tendrá la recompensa implícita de la opción preferida?
3. Si β fuera muy grande, ¿la política se movería más o menos respecto a la referencia?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
import math

def recompensa_implicita(p, p_ref, beta):
    return beta * (math.log(p) - math.log(p_ref))

pol = r['politica_dpo']
ref = r['politica_referencia']
for beta in (0.1, 0.5, 1.0):
    valores = {k: round(recompensa_implicita(pol[k], ref[k], beta), 3) for k in pol}
    print(f'β={beta:<4} → r̂ = {valores}')

## 9. Salida interpretable

La opción preferida tiene `r̂ > 0` y las rechazadas `r̂ < 0`: la política **es** el modelo de recompensa, leído como log-ratio contra la referencia. β escala esa recompensa y, en el objetivo, controla cuánto se permite alejarse de `π_ref`.


## 10. Comentario pedagógico

DPO es más simple, no automáticamente mejor. Sigue dependiendo por completo de la calidad y cobertura de los pares de preferencia, y no permite explorar respuestas nuevas fuera de la distribución de los datos, cosa que el muestreo on-policy de RLHF sí hace.


## 11. Error o anti-patrón deliberado

Anti-patrón: quitar `π_ref` de la fórmula porque «se simplifica». Sin referencia no hay restricción KL y la política colapsa.


In [ ]:
print('Si eliminas π_ref, la pérdida premia subir p(preferida) sin límite:')
for p in (0.5, 0.9, 0.99, 0.9999):
    print(f'  p={p:<7} → log p = {math.log(p):+.5f}  (nada frena el colapso a p→1)')
print('Resultado: una política degenerada que siempre dice lo mismo.')

## 12. Corrección

Con `π_ref` el término es un *log-ratio*: alejarse cuesta, y β pone el precio.


In [ ]:
p_ref = 0.27
for p in (0.5, 0.9, 0.99, 0.9999):
    ratio = math.log(p) - math.log(p_ref)
    print(f'  p={p:<7} → β·log(π/π_ref) con β=0.5 = {0.5 * ratio:+.4f}')

## 13. Desafío guiado

Comprueba la simetría: la suma de recompensas implícitas ponderadas se mantiene acotada.


In [ ]:
for semilla in (1, 7, 42):
    res = run_paper_lab('dpo', seed=semilla)['result']
    print(f"semilla {semilla:>2} · π_dpo = {res['politica_dpo']} "
          f"· pérdida final {res['perdida'][-1]['loss']}")

## 14. Desafío autónomo

Implementa DPO sobre un modelo de lenguaje pequeño y abierto con 200 pares de preferencia. Compara contra best-of-n con un modelo de recompensa entrenado sobre los mismos pares. Reporta preferencia humana ciega sobre 50 salidas y el coste de cómputo de cada vía.


## 15. Evidencia de aprendizaje

Guarda π_ref, π_DPO, la recompensa implícita para tres valores de β y tu explicación de por qué eliminar π_ref rompe el método.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P15_dpo/README.md) · evaluación formal: [`assessments/papers/P15_dpo.md`](../../assessments/papers/P15_dpo.md)


## 16. Cierre

Con alineación directa y herramientas autosupervisadas, las piezas del agente moderno están sobre la mesa. Queda ensamblarlas en un sistema.


## 17. Conexión con el siguiente hito

- P16

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
